# Cardio Catch Disease — 05. Deployment & Consumption

> **Hands-On ML principle (Ch. 2):** Deploy the model so others can use it. A data science project isn't done until the model is serving predictions. Document how to run it, monitor it, and update it.

---

## Deployment Architecture

```
┌─────────────────────────────────────────────────────┐
│                  Cardio Catch Disease               │
│                  Deployment Stack                   │
├──────────────────┬──────────────────────────────────┤
│  Training        │  Serving                         │
│  ─────────       │  ───────                         │
│  make train      │  FastAPI  (port 8000)             │
│  → model.joblib  │  ├─ GET  /health                 │
│                  │  └─ POST /predict                │
│                  │                                  │
│                  │  Streamlit (port 8501)            │
│                  │  └─ UI: individual patient scoring│
│                  │                                  │
│                  │  Google Sheets (optional)         │
│                  │  └─ Apps Script → API             │
└──────────────────┴──────────────────────────────────┘
```

All services are containerised with **Docker Compose** for one-command startup.

## 1. Train Model via CLI

In [ ]:
# Run this cell to train the model (or use `make train` in terminal)
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

result = subprocess.run(
    [sys.executable, "-m", "cardio_catch_disease.cli", "train"],
    env={**__import__("os").environ, "PYTHONPATH": str(PROJECT_ROOT / "src")},
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 2. Load Saved Model & Make Predictions

In [ ]:
import sys
from pathlib import Path
import joblib, json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from cardio_catch_disease.config import load_config
from cardio_catch_disease.features import prepare_features, model_matrix

config = load_config(PROJECT_ROOT / "configs" / "project.toml")
model  = joblib.load(PROJECT_ROOT / "models" / "model.joblib")

# Sample patient data for prediction
sample_patients = pd.DataFrame([
    # age(days), gender, height(cm), weight(kg), ap_hi, ap_lo, chol, gluc, smoke, alco, active
    {"age": 19000, "gender": 2, "height": 168, "weight": 85,  "ap_hi": 150, "ap_lo": 95,  "cholesterol": 2, "gluc": 1, "smoke": 0, "alco": 0, "active": 1},
    {"age": 14600, "gender": 1, "height": 162, "weight": 60,  "ap_hi": 120, "ap_lo": 80,  "cholesterol": 1, "gluc": 1, "smoke": 0, "alco": 0, "active": 1},
    {"age": 21900, "gender": 2, "height": 175, "weight": 110, "ap_hi": 180, "ap_lo": 110, "cholesterol": 3, "gluc": 2, "smoke": 1, "alco": 1, "active": 0},
])

# Apply same feature engineering as training
sample_prepared = prepare_features(sample_patients.copy(), config, training=False)
X_sample, _, _  = model_matrix(sample_prepared, config, training=False)

predictions = model.predict(X_sample)
probabilities = model.predict_proba(X_sample)[:, 1]

results = sample_patients.copy()
results["prediction"] = predictions
results["risk_score"]  = probabilities.round(3)
results["risk_level"]  = pd.cut(probabilities, bins=[0, 0.3, 0.6, 1.0],
                                  labels=["Low", "Medium", "High"])
print("Sample Predictions:")
results[["age", "ap_hi", "ap_lo", "cholesterol", "prediction", "risk_score", "risk_level"]]

## 3. Read Saved Metrics

In [ ]:
metrics_path = PROJECT_ROOT / "reports" / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print(json.dumps(metrics, indent=2))
else:
    print("metrics.json not found — run `make train` first.")

## 4. FastAPI — Start & Test the REST Endpoint

In [ ]:
# FastAPI endpoint usage — run this AFTER starting the API:
# Terminal: make api     (or: PYTHONPATH=src uvicorn cardio_catch_disease.api:app --reload)
import requests

API_URL = "http://127.0.0.1:8000"

# Health check
try:
    health = requests.get(f"{API_URL}/health", timeout=3)
    print("Health check:", health.json())
except requests.exceptions.ConnectionError:
    print("API not running. Start it with: make api")
    print("Then re-run this cell.")

In [ ]:
# Predict via API
payload = {
    "records": [
        {"age": 19000, "gender": 2, "height": 168, "weight": 85,
         "ap_hi": 150, "ap_lo": 95, "cholesterol": 2, "gluc": 1,
         "smoke": 0, "alco": 0, "active": 1},
        {"age": 14600, "gender": 1, "height": 162, "weight": 60,
         "ap_hi": 120, "ap_lo": 80, "cholesterol": 1, "gluc": 1,
         "smoke": 0, "alco": 0, "active": 1},
    ]
}

try:
    response = requests.post(f"{API_URL}/predict", json=payload, timeout=10)
    result = response.json()
    print("API Response:")
    print(json.dumps(result, indent=2))

    for i, (pred, score) in enumerate(zip(result["prediction"], result["score"])):
        label = "Disease detected" if pred == 1 else "No disease"
        print(f"Patient {i+1}: {label} — Risk score: {score:.1%}")

except requests.exceptions.ConnectionError:
    print("API not running. Start with: make api")

## 5. Docker Deployment (One Command)

The entire stack (API + Streamlit app) can be launched with Docker Compose:

```bash
# Build and start all services
docker compose up --build

# API available at:       http://localhost:8000
# API docs at:            http://localhost:8000/docs
# Streamlit app at:       http://localhost:8501
```

The Docker setup:
1. Installs all dependencies
2. Trains the model automatically on container start (if model not cached)
3. Serves the FastAPI endpoint
4. Runs the Streamlit app

```yaml
# docker-compose.yml overview
services:
  api:        # FastAPI — exposes /predict
  streamlit:  # Streamlit UI — calls the API
```

## 6. Google Sheets Integration

For non-technical users who prefer spreadsheets:

1. Deploy the FastAPI to a public URL (e.g. Render, Railway, or ngrok for local)
2. Open Google Sheets → Extensions → Apps Script
3. Paste the content of `integrations/google_sheets_appscript.gs`
4. Set `CARDIO_API_URL` to your public API URL
5. Use the **Cardio Catch** menu item to score the active sheet

The script reads patient data from selected rows and writes back risk scores automatically.

## 7. API Documentation

FastAPI auto-generates interactive docs at `http://localhost:8000/docs`:

| Endpoint | Method | Description |
|---|---|---|
| `/health` | GET | Returns `{"status": "ok"}` if API is live |
| `/predict` | POST | Accepts a list of patient records, returns predictions + risk scores |

**Request body schema (`/predict`):**
```json
{
  "records": [
    {
      "age": 19000,
      "gender": 2,
      "height": 168,
      "weight": 85,
      "ap_hi": 150,
      "ap_lo": 95,
      "cholesterol": 2,
      "gluc": 1,
      "smoke": 0,
      "alco": 0,
      "active": 1
    }
  ]
}
```

**Response schema:**
```json
{
  "prediction": [1],
  "score": [0.732]
}
```

## 8. Monitoring & Next Steps

**What to monitor in production:**
- **Data drift:** Distribution of incoming features shifting from training distribution
- **Prediction drift:** Score distribution shifting over time
- **Model degradation:** Accuracy dropping as population changes
- **API latency:** Ensure p95 response time stays < 200ms

**Recommended next steps:**

| Priority | Task |
|---|---|
| High | Add model versioning and a champion/challenger framework |
| High | Integrate SHAP for per-prediction explainability |
| Medium | Add threshold tuning UI in Streamlit (adjust recall/precision trade-off) |
| Medium | Set up automated retraining pipeline when new labelled data arrives |
| Low | Add demographic risk breakdown dashboard (age group, BMI band) |